# Stack v3 — Held-out No-FIM Python Pool (`stack_v3_python_only_data_without_fim`)
**Kaggle notebook — thin controller only. All logic lives in `scripts/build_sample.py` and `src/curation/`.**

> **PLAN CHANGE (2026-08-28).** This pool is no longer a fresh 50,000-file
> curation from the start of `stack-v3-train`. It now **resumes from the 10k
> pilot's final checkpoint** —
> [`sample_filtered_data_10000/checkpoints/checkpoint_chunk_0009.json`](https://huggingface.co/datasets/Rudra-G-23/the-stack-v3-python-fim-data/blob/main/sample_filtered_data_10000/checkpoints/checkpoint_chunk_0009.json)
> (`shard_index` 9, `row_offset` 13500, `cumulative_files_collected` 10000) —
> and keeps streaming from *that* point onward. Every file it collects is
> therefore strictly newer than anything the pilot scanned. We stop as soon
> as we have enough for the fixed `safim_eval_1000` eval set (`target.files`
> is now **2,000**, not 50,000); the first 13,500 rows are never
> re-streamed or re-filtered.

This notebook:
1. Clones the repo
2. Installs the (CPU-only) curation dependencies
3. Sets the HF token from Kaggle Secrets
4. Runs `scripts/build_sample.py` against `configs/data/stack_v3_eval_pool_no_fim.yaml`
   for one `--max-gb`-bounded session
5. Shows the auto-generated filter report

Target: 2,000 curated Python files (not the full corpus, and no longer 50k —
see `.claude/stages/data_stages/data-stage-2.md` §1 for why an explicit
finite ceiling is used instead of streaming uncapped). On this pool's very
first session, `continue_from_path_prefix` in the config makes it a strict
continuation of `sample_filtered_data_10000`: its dedup set is seeded from
that sample's already-complete hash set **and** streaming starts at
`(shard 9 + 1, row 13500)` instead of `(0, 0)`. So every file collected here
is disjoint from the original 10k pilot sample by stream position *and* by
content hash — not filtered after the fact.

No GPU needed for this stage. Resuming a later session picks up
automatically from the highest checkpoint already pushed to the HF dataset
repo — nothing to copy-paste between sessions. Re-run cell 4 as many times
(across as many sessions) as it takes to reach 2,000 files; `--max-gb`
bounds how much of that happens in any one session.


In [ ]:
# ── Cell 1: Clone repository at the requested Git state ──────────────────────────
import os
import shutil
import subprocess

GITHUB_REPO = "https://github.com/Rudra-G-23/qwen2.5-coder-0.5b-python-fim.git"

# Set these as needed
BRANCH = "feat/data"  # None -> main
COMMIT = None  # None -> latest commit on BRANCH

REPO_DIR = "/kaggle/working/qwen2.5-coder-0.5b-python-fim"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f"Cloning branch : {BRANCH or 'main'}")
print(f"Requested commit : {COMMIT or '(latest on branch)'}")

clone_cmd = ["git", "clone"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH]
clone_cmd += [GITHUB_REPO, REPO_DIR]
subprocess.run(clone_cmd, check=True)

if COMMIT:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", COMMIT], check=True)

current_branch = subprocess.check_output(
    ["git", "-C", REPO_DIR, "branch", "--show-current"], text=True
).strip()
current_commit = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True
).strip()

print("\n✓ Repository ready")
print(f"  Branch : {current_branch or '(detached HEAD)'}")
print(f"  Commit : {current_commit}")


In [ ]:
# ── Cell 2: Install dependencies (CPU-only, no torch/unsloth needed here) ───
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "datasets",
        "pyarrow",
        "huggingface_hub",
        "pyyaml",
        "wandb",
        "weave",
    ],
    check=True,
)


In [ ]:
# ── Cell 3: Authenticate to Hugging Face ────────────────────────
# HF_TOKEN is stored as a Kaggle Secret — NEVER hardcode tokens.
# Add it: Kaggle account → Settings → Secrets → Add New Secret
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")


In [ ]:
# ── Cell 4: Run one curation session ───────────────────────────
# PLAN CHANGE (2026-08-28): no longer a fresh 50k pull. This resumes the 10k
# pilot's stream from sample_filtered_data_10000/checkpoints/checkpoint_chunk_0009.json
# (shard 9, row 13500, 10000 files) and collects only what's needed for the
# fixed safim_eval_1000 eval set — target.files is 2,000 in the config below.
#
# --max-gb bounds this session's collected (post-filter) bytes — tune to what
# fits comfortably in the 12-hour Kaggle session cap. Re-run this same cell
# in later sessions to resume automatically from the last checkpoint, until
# target.files (2,000) is reached.
import os

os.chdir(REPO_DIR)

MAX_GB = 2

subprocess.run(
    [
        sys.executable,
        "scripts/build_sample.py",
        "--config", "configs/data/stack_v3_eval_pool_no_fim.yaml",
        "--max-gb", str(MAX_GB),
    ],
    check=True,
)


In [ ]:
# ── Cell 5: Show the auto-generated filter report ──────────────────
from IPython.display import Markdown, display

report_path = f"{REPO_DIR}/reports/stage1_filter_report.md"
with open(report_path, encoding="utf-8") as f:
    display(Markdown(f.read()))
